# RSNA Knee | DINOsaur V4 — TRAIN 🦖

Expected output folder: `/kaggle/working/rsna-knee-dinov3-slothead-v16/`.
Expected files: `v16_manifest.json`, `v16_slothead_f*.pt`, `v16_slothead_oof.csv`.


In [ ]:
from pathlib import Path
import os, gc, json, math, time, hashlib, random, warnings, copy
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
SEED = 20260820
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT = Path("/kaggle/working/rsna-knee-dinov3-slothead-v16")
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import gc, os, time, warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)
CROP_MM = 130.0
SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICE = 16
INTENSITY = 'slice'
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

def _find_dir(*names):
    root = Path('/kaggle/input')
    cand = []
    for n in names:
        cand += [root / n, root / 'competitions' / n, root / 'datasets' / n]
        for parent in (root / 'datasets', root / 'competitions', root):
            if parent.is_dir():
                try:
                    cand += [d / n for d in parent.iterdir() if d.is_dir()]
                except OSError:
                    pass
    for p in cand:
        if p.is_dir():
            return p
    return None
COMP = _find_dir('rsna-knee-abnormality-detection')
CKPT = _find_dir('knee-mri-fold-weights')
assert COMP is not None, 'competition data not attached'
assert CKPT is not None, 'fold weights not attached'
assert (COMP / 'sample_submission.csv').exists(), f'no competition data at {COMP}'
assert list(CKPT.glob('*_f*.pt')), f'no checkpoints at {CKPT}'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'competition : {COMP}')
print(f'checkpoints : {CKPT}')
print(f'device      : {DEV}')
for i in range(torch.cuda.device_count() if DEV == 'cuda' else 0):
    cc = torch.cuda.get_device_capability(i)
    print(f'  gpu{i}       : {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]}, {torch.cuda.get_device_properties(i).total_memory / 2 ** 30:.0f} GiB, native bf16={cc >= (8, 0)}')


In [ ]:
SERIES_ROOT = COMP / 'test_series'
if not SERIES_ROOT.exists():
    SERIES_ROOT = COMP / 'train_series'
print('series root:', SERIES_ROOT)

def ordered_files(sdir, cap=64):
    keyed = []
    for f in sdir.glob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            keyed.append((int(ds.InstanceNumber), str(f)))
        except Exception:
            continue
        if len(keyed) >= cap * 4:
            break
    return [f for _, f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(args):
    idx, study, recs = args
    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    if len(rows):
        for s_i, (plane, fs) in enumerate(SLOTS):
            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
            if sub.empty:
                continue
            files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)
            if not files:
                continue
            flip = plane != 'Sagittal' and series_side(files[0]) < 0
            lo, hi = SLICE_BAND
            i0 = int(round(lo * (len(files) - 1)))
            i1 = int(round(hi * (len(files) - 1)))
            avail = list(range(i0, i1 + 1))
            if len(avail) >= N_SLICE:
                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                off = 0
            else:
                picks, off = (avail, (N_SLICE - len(avail)) // 2)
            if INTENSITY == 'series':
                crops = [read_crop(files[p]) for p in picks]
                got = [x for x in crops if x is not None]
                if got:
                    samp = np.concatenate([x[::4, ::4].ravel() for x in got])
                    lo_, hi_ = np.percentile(samp, [1, 99])
                    for c, x in enumerate(crops):
                        if x is None:
                            x = read_crop(files[min(len(files) - 1, picks[c] + 1)])
                        if x is not None:
                            out[s_i, off + c] = (window(x, lo_, hi_, flip) * 255).astype(np.uint8)
            else:
                for c, p in enumerate(picks):
                    img = render(files[p], flip)
                    if img is None:
                        img = render(files[min(len(files) - 1, p + 1)], flip)
                    if img is not None:
                        out[s_i, off + c] = (img * 255).astype(np.uint8)
            mask[s_i] = len(picks)
    return (idx, out, mask)


In [ ]:
N_SLOT_TYPES, MASK_IDX = (6, 0)

def segment_softmax(scores, sidx, B):
    T, K = scores.shape
    idx = sidx.unsqueeze(1).expand(-1, K)
    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
    e = (scores - m[sidx]).exp()
    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
    return e / s[sidx].clamp(min=1e-06)

class MeanMaxPool(nn.Module):

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        D = f.shape[1]
        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
        mean = mean / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
        return (torch.cat([mean, mx], 1), None)

class LabelAttentionPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = (d, n_labels, n_heads)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        scores = self.key(f) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        a = segment_softmax(scores, sidx, B)
        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
        return (out, a)

class TokenXAttnPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = (d, n_labels)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tok, sidx, B, slot=None, return_attn=False):
        T, N, D = tok.shape
        cnt = torch.bincount(sidx, minlength=B)
        S = int(cnt.max().item())
        starts = torch.cumsum(cnt, 0) - cnt
        pos = torch.arange(T, device=tok.device) - starts[sidx]
        kv = tok + self.slot_emb(slot).unsqueeze(1)
        pad = tok.new_zeros(B, S, N, D)
        pad[sidx, pos] = kv
        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
        keep[sidx, pos] = True
        kpm = ~keep.repeat_interleave(N, dim=1)
        pad = self.kv_norm(pad.reshape(B, S * N, D))
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        cls = tok[:, 0]
        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
        return (torch.cat([att, base], -1), w)

class ViTSlotToken(nn.Module):

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        d = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for blk in vit.blocks:
            a = getattr(blk, 'attn', None)
            if a is not None and hasattr(a, 'num_prefix_tokens'):
                a.num_prefix_tokens = a.num_prefix_tokens + 1

    @staticmethod
    def _maybe(mod, x):
        return x if mod is None else mod(x)

    def forward_features(self, x, cat):
        v = self.vit
        x = v.patch_embed(x)
        pos = v._pos_embed(x)
        rope = None
        if isinstance(pos, tuple):
            x, rope = pos
        else:
            x = pos
        x = self._maybe(getattr(v, 'patch_drop', None), x)
        x = self._maybe(getattr(v, 'norm_pre', None), x)
        npt = self._orig_prefix
        tok = self.tok(cat).unsqueeze(1)
        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
        if rope is not None:
            if getattr(v, 'rope_mixed', False):
                for i, blk in enumerate(v.blocks):
                    x = blk(x, rope=rope[i])
            else:
                for blk in v.blocks:
                    x = blk(x, rope=rope)
        else:
            x = v.blocks(x)
        return v.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class _GatedDepthBlock(nn.Module):

    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

class DepthCompress(nn.Module):

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for b in self.blocks:
            z = b(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep
N_PLANE, N_CONTRAST = (3, 2)
_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

class SlotDepthMixer(nn.Module):

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
        self.alpha_max = alpha_max
        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer('base', b.log()[self.r:])
        n_u = self.r + 1
        self.shared = nn.Parameter(torch.zeros(n_u))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        idx = torch.arange(n_slice)
        self.register_buffer('off', idx[None, :] - idx[:, None])

    def kernel(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

    def forward(self, x, slot, vmask):
        T, S, H, W = x.shape
        if vmask is None:
            raise ValueError('stem=mixer requires the padding mask')
        k = self.kernel(slot)
        v = vmask.to(k.dtype)
        d = self.off + self.r
        inb = (d >= 0) & (d < self.ksize)
        kk = k[:, d.clamp(0, self.ksize - 1)] * inb
        M = kk * v[:, None, :]
        den = M.sum(-1, keepdim=True)
        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
        ok = (den > 1e-06) & v[:, :, None].bool()
        M = torch.where(ok, M / den.clamp(min=1e-06), eye)
        a = self.alpha(slot)[:, None, None]
        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

def _seg_mean_max(v, sidx, B):
    D = v.shape[1]
    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
    mean = mean / cnt.clamp(min=1).unsqueeze(1)
    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
    return torch.cat([mean, mx], 1)

def _pad_kv(x, sidx, B, norm):
    T, P, D = x.shape
    cnt = torch.bincount(sidx, minlength=B)
    S = int(cnt.max().item())
    starts = torch.cumsum(cnt, 0) - cnt
    pos = torch.arange(T, device=x.device) - starts[sidx]
    pad = x.new_zeros(B, S, P, D)
    pad[sidx, pos] = x
    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
    keep[sidx, pos] = True
    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

class _GatedDelta(nn.Module):

    def __init__(self, d, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(d)
        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, pat, sidx, B, return_attn):
        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

class TokenResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class CodexResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class ClsAddPool(nn.Module):

    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

class Readout(nn.Module):

    def __init__(self, pool, d, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = (pool, n_labels)
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ('xres', 'clsadd', 'xcodex'):
            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
        elif pool in ('attn', 'xattn'):
            if pool == 'xattn':
                self.pool = TokenXAttnPool(d, n_labels)
                wd = 3 * d + pe
            else:
                self.pool = LabelAttentionPool(d, n_labels)
                wd = d + pe
            self.norm = nn.LayerNorm(wd)
            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, f, slot, sidx, B, return_attn=False):
        pe = self.pres_emb(slot)
        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
            return self.pool(f, slot, sidx, B, pres)[0]
        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
        if self.pool_kind in ('attn', 'xattn'):
            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, pres], 1))

class Net(nn.Module):

    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
        super().__init__()
        self.enc, self.cond = (enc, cond)
        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
        D = enc.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
        self.readout = Readout(pool, D)
        if cond == 'post':
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

    def forward(self, im, slot, smeta, sidx, B, vm=None):
        if self.mixer is not None:
            im = self.mixer(im, slot, vm)
        if self.compress is not None:
            im = self.compress(im)
        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
        if self.tokens:
            inner = getattr(self.enc, 'vit', self.enc)
            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
            f = torch.cat([f[:, :1], f[:, orig:]], 1)
        else:
            f = self.enc.forward_head(f, pre_logits=True)
            if f.dim() > 2:
                f = f.flatten(1)
        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
        if self.cond == 'post':
            f = f + ex(self.slot_emb(slot))
        if self.meta_mlp is not None and smeta.shape[1] > 0:
            mt = self.meta_mlp(smeta)
            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
        return self.readout(f, slot, sidx, B)
models = []
for ckpt_path in sorted(CKPT.glob('*_f*.pt')):
    z = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = z['cfg']
    _stem = cfg.get('stem', 'native')
    _in = 3 if _stem == 'compress' else cfg.get('n_slice', 16)
    enc = timm.create_model(cfg['backbone'], pretrained=False, num_classes=0, in_chans=_in, **{'img_size': cfg['img']} if 'vit_' in cfg['backbone'] else {})
    if cfg['cond'] == 'token':
        enc = ViTSlotToken(enc, N_SLOT_TYPES)
    m = Net(enc, cfg['cond'], cfg.get('n_meta', 0), cfg['pool'], stem=_stem, n_slice=cfg.get('n_slice', 16))
    missing, unexpected = m.load_state_dict(z['state_dict'], strict=False)
    assert not [k for k in missing if not k.startswith('enc.')], f'missing {missing[:5]}'
    assert not unexpected, f'unexpected {unexpected[:5]}'
    models.append(m.eval())
    print(f"loaded {ckpt_path.name}  fold {z['fold']}  {cfg['backbone']} pool={cfg['pool']} meta={cfg['meta']}")
CFG = cfg
assert CFG.get('n_meta', 0) == 0, f"checkpoint expects {CFG['n_meta']} metadata features -- build slot_meta for the TEST studies and pass it to predict() before submitting"
print(f"\n{len(models)} fold models ready | input norm: {CFG.get('norm', 'none')}")


In [ ]:
# -------------------- V16 frozen DINOv3 feature cache --------------------

SERIES_ROOT = COMP / "train_series"
train_df = pd.read_csv(
    COMP / "train.csv",
    dtype={"StudyInstanceUID": str},
)
train_series = pd.read_csv(
    COMP / "train_series.csv",
    dtype={
        "StudyInstanceUID": str,
        "SeriesInstanceUID": str,
    },
)
train_series = train_series.loc[:, ~train_series.columns.duplicated()]
train_ids = train_df["StudyInstanceUID"].astype(str).tolist()
train_by = {
    str(uid): group.to_dict("records")
    for uid, group in train_series[
        train_series["StudyInstanceUID"].astype(str).isin(set(train_ids))
    ].groupby("StudyInstanceUID")
}

def _v16_find_file(filename):
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = [
                d for d in dirs
                if d not in ("train_series", "test_series", ".git", "__pycache__")
            ]
            if filename in files:
                matches.append(Path(current) / filename)
    matches = [p for p in matches if p.is_file()]
    matches.sort(key=lambda p: (len(p.parts), str(p)))
    return matches[0] if matches else None

def _v16_load_supervision():
    oof_path = _v16_find_file("oof.npz")
    fold_path = _v16_find_file("v52_e11_oof.csv")
    public_path = _v16_find_file("v52_oof.csv")

    if oof_path is None or fold_path is None or public_path is None:
        raise FileNotFoundError(
            "V16 needs oof.npz + v52_oof.csv + v52_e11_oof.csv as Kaggle inputs"
        )

    with np.load(oof_path, allow_pickle=False) as data:
        ids = data["ids"].astype(str)
        targets = data["targets"].astype(str).tolist()
        weak_y = data["y_derived"].astype(np.float32)
        base_oof = data["pred"].astype(np.float32)
        gold = data["gold_mask"].astype(bool)

    if targets != LABELS:
        raise RuntimeError("OOF target contract mismatch")
    if not np.array_equal(ids, np.asarray(train_ids, dtype=str)):
        raise RuntimeError("OOF train ID order mismatch")

    exact = train_df[LABELS].apply(pd.to_numeric, errors="coerce")
    official_gold = exact.notna().all(axis=1).to_numpy()
    if not np.array_equal(gold, official_gold):
        raise RuntimeError("gold_mask mismatch")

    y = np.where(np.isfinite(weak_y), weak_y, 0.5).astype(np.float32)
    y[gold] = exact.loc[gold, LABELS].to_numpy(np.float32)

    fold_frame = pd.read_csv(
        fold_path,
        dtype={"StudyInstanceUID": str},
    )
    fold_frame = train_df[["StudyInstanceUID"]].merge(
        fold_frame[["StudyInstanceUID", "fold"] + LABELS],
        on="StudyInstanceUID",
        how="left",
        validate="one_to_one",
    )
    fold = pd.to_numeric(fold_frame["fold"], errors="coerce").to_numpy()
    if not np.isfinite(fold).all():
        raise RuntimeError("OOF fold assignment incomplete")
    fold = fold.astype(np.int64)

    public_frame = pd.read_csv(
        public_path,
        dtype={"StudyInstanceUID": str},
    )
    public_frame = train_df[["StudyInstanceUID"]].merge(
        public_frame[["StudyInstanceUID"] + LABELS],
        on="StudyInstanceUID",
        how="left",
        validate="one_to_one",
    )

    def rank_cols(a):
        return pd.DataFrame(a).rank(method="average", pct=True).to_numpy(np.float64)

    base_rank = rank_cols(base_oof)
    public_rank = rank_cols(public_frame[LABELS].to_numpy(np.float64))
    pass2_rank = rank_cols(fold_frame[LABELS].to_numpy(np.float64))
    approx_anchor = rank_cols(
        0.85 * rank_cols(0.50 * base_rank + 0.50 * public_rank)
        + 0.15 * pass2_rank
    )

    return y, gold, fold, approx_anchor, str(oof_path)

Y, GOLD, FOLD, APPROX_ANCHOR, OOF_SOURCE = _v16_load_supervision()

# The encoder is exactly the already attached DINOv3 fold-0 model.
encoder_model = models[0].eval()
encoder_cfg = dict(CFG)
encoder_ckpt = sorted(CKPT.glob("*_f*.pt"))[0]

DEV_FEAT = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder_model = encoder_model.to(DEV_FEAT)
AMP_DTYPE = torch.float16 if str(DEV_FEAT).startswith("cuda") else torch.float32

def _v16_norm(im):
    kind = encoder_cfg.get("norm", "none")
    if kind == "zscore":
        m = (im > 0).float()
        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
        return (im - mu) / (var.sqrt() + 1e-6) * m
    if kind == "imagenet":
        m = (im > 0).float()
        return (im - 0.485) / 0.229 * m
    return im

@torch.no_grad()
def _v16_encode_slots(images, masks, micro=8):
    B = len(masks)
    records = []

    for study_index in range(B):
        present = np.nonzero(masks[study_index] > 0)[0]
        for slot in present:
            records.append((study_index, int(slot)))

    if not records:
        return None, np.zeros((B, N_SLOT), np.uint8)

    feature_batches = []
    index_batches = []

    for start in range(0, len(records), micro):
        part = records[start:start + micro]
        ims = np.stack(
            [images[study_index, slot] for study_index, slot in part],
            axis=0,
        )
        im = torch.from_numpy(ims).to(DEV_FEAT).float().div_(255.0)
        im = _v16_norm(im)

        slots = torch.tensor(
            [slot + 1 for _, slot in part],
            device=DEV_FEAT,
            dtype=torch.long,
        )
        vm = torch.from_numpy(
            ims.reshape(len(ims), ims.shape[1], -1).max(2) > 0
        ).to(DEV_FEAT)

        model = encoder_model
        x = im
        if model.mixer is not None:
            x = model.mixer(x, slots, vm)
        if model.compress is not None:
            x = model.compress(x)

        with torch.autocast(
            "cuda" if str(DEV_FEAT).startswith("cuda") else "cpu",
            dtype=AMP_DTYPE,
            enabled=str(DEV_FEAT).startswith("cuda"),
        ):
            if encoder_cfg["cond"] == "token":
                tokens = model.enc.forward_features(x, slots)
            else:
                tokens = model.enc.forward_features(x)

        if tokens.ndim == 2:
            cls = tokens.float()
            mean = tokens.float()
            focal = tokens.float()
        else:
            inner = getattr(model.enc, "vit", model.enc)
            prefix = getattr(
                model.enc,
                "_orig_prefix",
                getattr(inner, "num_prefix_tokens", 1),
            )
            cls = tokens[:, 0].float()
            patches = tokens[:, prefix:].float()
            mean = patches.mean(dim=1)

            n_patch = patches.shape[1]
            grid = int(round(math.sqrt(n_patch)))
            if grid * grid == n_patch and grid >= 3:
                p = patches.reshape(len(part), grid, grid, -1)
                radius = max(1, grid // 4)
                center = grid // 2
                y0, y1 = max(0, center - radius), min(grid, center + radius + 1)
                focal = p[:, y0:y1, y0:y1].mean(dim=(1, 2))
            else:
                focal = mean

        feat = torch.cat([cls, mean, focal], dim=1).cpu().numpy().astype(np.float16)
        feature_batches.append(feat)
        index_batches.extend(part)

    feature_dim = feature_batches[0].shape[1]
    out = np.zeros((B, N_SLOT, feature_dim), np.float16)
    out_mask = np.zeros((B, N_SLOT), np.uint8)

    offset = 0
    for batch in feature_batches:
        for row in range(len(batch)):
            study_index, slot = index_batches[offset]
            out[study_index, slot] = batch[row]
            out_mask[study_index, slot] = 1
            offset += 1

    return out, out_mask

FEATURE_CACHE = OUT / "train_features_fp16.npz"

if FEATURE_CACHE.exists():
    cache = np.load(FEATURE_CACHE, allow_pickle=False)
    FEATURES = cache["features"]
    SLOT_MASK = cache["mask"]
    cached_ids = cache["ids"].astype(str)
    if not np.array_equal(cached_ids, np.asarray(train_ids, dtype=str)):
        raise RuntimeError("feature cache ID mismatch")
    print("loaded feature cache", FEATURES.shape)
else:
    WORKERS = max(1, min(6, os.cpu_count() or 4))
    CHUNK = 24
    chunks = []
    mask_chunks = []

    t0 = time.time()
    with ProcessPoolExecutor(max_workers=WORKERS) as ex:
        for c0 in range(0, len(train_ids), CHUNK):
            block = train_ids[c0:c0 + CHUNK]
            imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
            msks = np.zeros((len(block), N_SLOT), np.uint8)

            futures = [
                ex.submit(build_study, (i, uid, train_by.get(uid, [])))
                for i, uid in enumerate(block)
            ]
            for future in as_completed(futures):
                try:
                    i, arr, mask = future.result()
                    imgs[i], msks[i] = arr, mask
                except Exception as exc:
                    print("decode failed", type(exc).__name__, exc)

            feat, feat_mask = _v16_encode_slots(imgs, msks, micro=8)
            if feat is None:
                raise RuntimeError("empty feature block")

            chunks.append(feat)
            mask_chunks.append(feat_mask)

            done = min(c0 + len(block), len(train_ids))
            elapsed = time.time() - t0
            print(
                f"features {done}/{len(train_ids)} "
                f"{elapsed/60:.1f}m "
                f"eta={elapsed/max(done,1)*(len(train_ids)-done)/60:.1f}m",
                flush=True,
            )

            del imgs, msks, feat, feat_mask
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    FEATURES = np.concatenate(chunks, axis=0)
    SLOT_MASK = np.concatenate(mask_chunks, axis=0)

    np.savez_compressed(
        FEATURE_CACHE,
        ids=np.asarray(train_ids, dtype="U80"),
        features=FEATURES,
        mask=SLOT_MASK,
    )
    print("saved feature cache", FEATURES.shape)

encoder_model = encoder_model.cpu()
del models
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# -------------------- V16 target-conditioned slot-attention head --------------------

class SlotAttentionHead(nn.Module):
    def __init__(self, feature_dim, hidden=384, n_target=12, n_slot=6):
        super().__init__()
        self.feature_dim = int(feature_dim)
        self.hidden = int(hidden)
        self.n_target = int(n_target)
        self.n_slot = int(n_slot)

        self.proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden),
            nn.GELU(),
        )
        self.slot_emb = nn.Parameter(torch.zeros(n_slot, hidden))
        nn.init.normal_(self.slot_emb, std=0.02)

        self.query = nn.Parameter(torch.zeros(n_target, hidden))
        nn.init.normal_(self.query, std=0.02)

        self.attn = nn.MultiheadAttention(
            hidden,
            num_heads=6,
            dropout=0.10,
            batch_first=True,
        )

        self.fuse = nn.Sequential(
            nn.LayerNorm(hidden * 5),
            nn.Linear(hidden * 5, hidden),
            nn.GELU(),
            nn.Dropout(0.16),
        )

        self.target_weight = nn.Parameter(torch.empty(n_target, hidden))
        self.target_bias = nn.Parameter(torch.zeros(n_target))
        nn.init.normal_(self.target_weight, std=0.03)

    def forward(
        self,
        features,
        slot_mask,
        modality_dropout=0.0,
        feature_noise=0.0,
    ):
        mask = slot_mask.bool()

        if self.training and modality_dropout > 0:
            keep = torch.rand_like(slot_mask.float()) > float(modality_dropout)
            keep = keep & mask
            empty = keep.sum(dim=1) == 0
            if empty.any():
                first = mask[empty].float().argmax(dim=1)
                keep[empty] = False
                keep[empty, first] = True
            mask = keep

        x = self.proj(features.float())
        x = x + self.slot_emb.unsqueeze(0)

        if self.training and feature_noise > 0:
            x = x + torch.randn_like(x) * float(feature_noise)

        x = x * mask.unsqueeze(-1)
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1).unsqueeze(-1)
        mean = x.sum(dim=1, keepdim=True) / denom

        query = self.query.unsqueeze(0).expand(len(x), -1, -1)
        attended, _ = self.attn(
            query,
            x,
            x,
            key_padding_mask=~mask,
            need_weights=False,
        )

        global_target = mean.expand(-1, self.n_target, -1)
        fused = torch.cat(
            [
                attended,
                query,
                global_target,
                torch.abs(attended - global_target),
                attended * global_target,
            ],
            dim=-1,
        )
        fused = self.fuse(fused)
        logits = (
            fused * self.target_weight.unsqueeze(0)
        ).sum(dim=-1) + self.target_bias.unsqueeze(0)
        return logits


def _macro_auc(y_true, pred):
    aucs = []
    for j in range(len(LABELS)):
        y = y_true[:, j]
        p = pred[:, j]
        if len(np.unique(y)) == 2:
            aucs.append(roc_auc_score(y, p))
    return float(np.mean(aucs)) if aucs else float("nan")


def _ema_update(ema, model, decay=0.995):
    with torch.no_grad():
        ema_state = dict(ema.named_parameters())
        for name, parameter in model.named_parameters():
            ema_state[name].mul_(decay).add_(
                parameter.detach(),
                alpha=1.0 - decay,
            )
        ema_buffers = dict(ema.named_buffers())
        for name, buffer in model.named_buffers():
            ema_buffers[name].copy_(buffer)


def _batch_rank_loss(logits, labels, gold_rows):
    losses = []
    for target in range(logits.shape[1]):
        y = labels[:, target]
        positive = torch.nonzero(gold_rows & (y > 0.5), as_tuple=False).flatten()
        negative = torch.nonzero(gold_rows & (y <= 0.5), as_tuple=False).flatten()
        if len(positive) == 0 or len(negative) == 0:
            continue

        n = min(24, len(positive), len(negative))
        pos = positive[torch.randperm(len(positive), device=logits.device)[:n]]
        neg = negative[torch.randperm(len(negative), device=logits.device)[:n]]
        losses.append(
            F.softplus(
                -(logits[pos, target] - logits[neg, target])
            ).mean()
        )
    if not losses:
        return logits.new_tensor(0.0)
    return torch.stack(losses).mean()


def _predict_head(model, indices, device, batch=512):
    model.eval()
    output = []
    with torch.no_grad():
        for start in range(0, len(indices), batch):
            idx = indices[start:start + batch]
            feat = torch.from_numpy(FEATURES[idx]).to(device)
            mask = torch.from_numpy(SLOT_MASK[idx]).to(device)
            logits = model(feat, mask)
            output.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(output, axis=0)


def _train_one_fold(fold_id, device):
    valid = GOLD & (FOLD == int(fold_id))
    if not valid.any():
        raise RuntimeError(f"fold {fold_id} has no gold validation rows")

    # Held-out gold is excluded even as pseudo-label supervision.
    allowed = ~valid
    gold_train = GOLD & allowed

    feature_dim = FEATURES.shape[-1]
    model = SlotAttentionHead(feature_dim).to(device)
    ema = copy.deepcopy(model).to(device).eval()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=7e-4,
        weight_decay=2e-3,
    )

    # v17: hard-denoise — zero-weight ambiguous weak labels (|Y-0.5|<0.15),
    # full weight for confident; replaces linear conf ramp.
    _conf = np.abs(Y - 0.5) * 2.0
    weak_conf = np.where(_conf < 0.15, 0.02,
                         np.where(_conf < 0.4, 0.8, 2.0)).astype(np.float32)

    target_weight = weak_conf.copy()
    target_weight[gold_train] = 14.0
    target_weight[valid] = 0.0

    base_pool = np.flatnonzero(allowed)
    gold_pool = np.flatnonzero(gold_train)

    # 32 weak-label epochs are cheap because the DINOv3 encoder is frozen.
    for epoch in range(32):
        repeated_gold = np.repeat(gold_pool, 8)
        pool = np.concatenate([base_pool, repeated_gold])
        rng = np.random.default_rng(SEED + 101 * fold_id + epoch)
        rng.shuffle(pool)

        model.train()
        running = 0.0
        steps = 0

        for start in range(0, len(pool), 256):
            idx = pool[start:start + 256]
            if len(idx) < 16:
                continue

            feat = torch.from_numpy(FEATURES[idx]).to(device)
            mask = torch.from_numpy(SLOT_MASK[idx]).to(device)
            labels = torch.from_numpy(Y[idx]).to(device)
            weights = torch.from_numpy(target_weight[idx]).to(device)
            gold_rows = torch.from_numpy(GOLD[idx]).to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(
                feat,
                mask,
                modality_dropout=0.13,
                feature_noise=0.012,
            )

            bce = F.binary_cross_entropy_with_logits(
                logits,
                labels,
                reduction="none",
            )
            loss = (bce * weights).sum() / weights.sum().clamp(min=1.0)
            loss = loss + 0.055 * _batch_rank_loss(
                logits,
                labels,
                gold_rows,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
            _ema_update(ema, model, 0.995)

            running += float(loss.detach().cpu())
            steps += 1

        if (epoch + 1) % 8 == 0 or epoch == 0:
            pred = _predict_head(
                ema,
                np.flatnonzero(valid),
                device,
            )
            score = _macro_auc(
                Y[valid],
                pred,
            )
            print(
                f"fold={fold_id} weak epoch={epoch+1:02d}/32 "
                f"loss={running/max(steps,1):.4f} gold_auc={score:.5f}"
            )

    # Exact-only refinement: small LR, stronger dropout, no held-out fold.
    exact_idx = np.flatnonzero(gold_train)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1.5e-4,
        weight_decay=3e-3,
    )

    for epoch in range(12):
        rng = np.random.default_rng(SEED + 9000 + 31 * fold_id + epoch)
        pool = np.tile(exact_idx, 10)
        rng.shuffle(pool)

        model.train()
        for start in range(0, len(pool), 128):
            idx = pool[start:start + 128]
            feat = torch.from_numpy(FEATURES[idx]).to(device)
            mask = torch.from_numpy(SLOT_MASK[idx]).to(device)
            labels = torch.from_numpy(Y[idx]).to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(
                feat,
                mask,
                modality_dropout=0.09,
                feature_noise=0.008,
            )
            loss = F.binary_cross_entropy_with_logits(
                logits,
                labels,
            ) + 0.10 * _batch_rank_loss(
                logits,
                labels,
                torch.ones(len(idx), dtype=torch.bool, device=device),
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.5)
            optimizer.step()
            _ema_update(ema, model, 0.997)

    valid_idx = np.flatnonzero(valid)
    pred = _predict_head(ema, valid_idx, device)

    checkpoint = {
        "version": 16,
        "fold": int(fold_id),
        "labels": list(LABELS),
        "feature_dim": int(feature_dim),
        "hidden": 384,
        "n_slot": int(N_SLOT),
        "state_dict": {
            key: value.detach().cpu()
            for key, value in ema.state_dict().items()
        },
    }

    path = OUT / f"v16_slothead_f{fold_id}.pt"
    torch.save(checkpoint, path)
    return valid_idx, pred, path


unique_folds = sorted(
    int(value)
    for value in np.unique(FOLD[GOLD])
    if int(value) >= 0
)
if len(unique_folds) < 4:
    raise RuntimeError("V16 expected at least four gold folds")

OOF_HEAD = np.full_like(Y, np.nan, dtype=np.float32)
saved = []

# The head is small; a single GPU is enough after feature extraction.
DEV_HEAD = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

for fold_id in unique_folds:
    valid_idx, pred, path = _train_one_fold(
        fold_id,
        DEV_HEAD,
    )
    OOF_HEAD[valid_idx] = pred
    saved.append(str(path))

gold_idx = np.flatnonzero(GOLD)
if not np.isfinite(OOF_HEAD[gold_idx]).all():
    raise RuntimeError("V16 gold OOF is incomplete")

print("V16 raw gold macro AUC:", _macro_auc(Y[GOLD], OOF_HEAD[GOLD]))


In [ ]:
# -------------------- nested blend selection + dataset manifest --------------------

def _rank_vector(values):
    return pd.Series(values).rank(method="average", pct=True).to_numpy(np.float64)

def _auc(y, p):
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, p))

def _nested_weight(truth, anchor, specialist, folds, target_index):
    grid = [0.00, 0.04, 0.08, 0.12, 0.16, 0.20, 0.24, 0.28]
    nested = np.full(len(truth), np.nan, np.float64)

    for outer in sorted(np.unique(folds)):
        choose = folds != outer
        valid = folds == outer
        best = 0.0
        best_obj = -1e9

        for weight in grid:
            pred = (1.0 - weight) * anchor[choose] + weight * specialist[choose]
            auc = _auc(truth[choose], pred)
            if not np.isfinite(auc):
                continue
            obj = auc - 0.0020 * weight
            if obj > best_obj:
                best_obj = obj
                best = weight

        nested[valid] = (1.0 - best) * anchor[valid] + best * specialist[valid]

    if not np.isfinite(nested).all():
        return 0.0, 0.0, 0.0

    base_auc = _auc(truth, anchor)
    nested_auc = _auc(truth, nested)
    gain = nested_auc - base_auc

    positive = np.flatnonzero(truth > 0.5)
    negative = np.flatnonzero(truth <= 0.5)
    rng = np.random.default_rng(16000 + target_index)
    gains = []

    if len(positive) >= 2 and len(negative) >= 2:
        for _ in range(320):
            idx = np.concatenate(
                [
                    rng.choice(positive, len(positive), replace=True),
                    rng.choice(negative, len(negative), replace=True),
                ]
            )
            a0 = _auc(truth[idx], anchor[idx])
            a1 = _auc(truth[idx], nested[idx])
            if np.isfinite(a0) and np.isfinite(a1):
                gains.append(a1 - a0)

    support = float((np.asarray(gains) > 0).mean()) if gains else 0.0

    # Final deployment weight is chosen on all gold only after nested validation.
    best_weight = 0.0
    best_obj = base_auc
    for weight in grid:
        pred = (1.0 - weight) * anchor + weight * specialist
        auc = _auc(truth, pred)
        obj = auc - 0.0020 * weight
        if obj > best_obj:
            best_obj = obj
            best_weight = weight

    protected = {
        "ACL",
        "Medial OA",
        "Lateral OA",
        "PF OA",
        "Effusion",
        "Baker's",
        "Contusion",
    }
    cap = 0.10 if LABELS[target_index] in protected else 0.22

    if gain >= 0.004 and support >= 0.62:
        deploy = min(best_weight, cap)
    elif gain >= 0.002 and support >= 0.59:
        deploy = min(best_weight, cap * 0.5)
    else:
        deploy = 0.0

    return float(deploy), float(gain), float(support)


head_gold = np.zeros((len(gold_idx), len(LABELS)), np.float64)
for j in range(len(LABELS)):
    head_gold[:, j] = _rank_vector(OOF_HEAD[gold_idx, j])

anchor_gold = APPROX_ANCHOR[gold_idx]
truth_gold = Y[gold_idx]
fold_gold = FOLD[gold_idx]

deploy_weights = {}
validation = {}

for j, target in enumerate(LABELS):
    weight, gain, support = _nested_weight(
        truth_gold[:, j],
        anchor_gold[:, j],
        head_gold[:, j],
        fold_gold,
        j,
    )
    deploy_weights[target] = weight
    validation[target] = {
        "nested_gain": gain,
        "bootstrap_support": support,
        "deploy_weight": weight,
        "head_auc": _auc(truth_gold[:, j], head_gold[:, j]),
        "anchor_auc": _auc(truth_gold[:, j], anchor_gold[:, j]),
    }

manifest = {
    "artifact_type": "rsna-knee-dinov3-slothead-v16",
    "version": 16,
    "labels": list(LABELS),
    "slots": [list(slot) for slot in SLOTS],
    "image_size": int(SIZE),
    "n_slice": int(N_SLICE),
    "crop_mm": float(CROP_MM),
    "feature": "per-slot CLS+mean+central-patch-mean from existing DINOv3 encoder",
    "encoder_checkpoint": encoder_ckpt.name,
    "encoder_cfg": encoder_cfg,
    "encoder_checkpoint_sha256": hashlib.sha256(
        encoder_ckpt.read_bytes()
    ).hexdigest(),
    "folds": [int(v) for v in unique_folds],
    "deploy_weights": deploy_weights,
    "inference_shrink": 0.85,
    "validation": validation,
    "oof_source": OOF_SOURCE,
}

(OUT / "v16_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

oof_frame = train_df[["StudyInstanceUID"]].copy()
for j, target in enumerate(LABELS):
    oof_frame[target] = OOF_HEAD[:, j]
oof_frame["gold"] = GOLD.astype(np.uint8)
oof_frame["fold"] = FOLD
oof_frame.to_csv(
    OUT / "v16_slothead_oof.csv",
    index=False,
)

# Remove the large train feature cache from the final weights artifact.
# It is useful during training but is not needed by inference.
try:
    FEATURE_CACHE.unlink()
except OSError:
    pass

import zipfile
zip_path = Path("/kaggle/working/rsna-knee-dinov3-slothead-v16.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print("saved weights:", OUT)
print("zip:", zip_path)
print("deploy weights:")
for target in LABELS:
    info = validation[target]
    print(
        f"{target:18s} weight={deploy_weights[target]:.3f} "
        f"gain={info['nested_gain']:+.4f} "
        f"support={info['bootstrap_support']:.3f}"
    )


In [ ]:
# V5.5 TRAIN output contract check
from pathlib import Path as _V55Path
import json as _v55_json

_V55_OUT = _V55Path('/kaggle/working/rsna-knee-dinov3-slothead-v16')
_V55_MANIFEST = _V55_OUT / 'v16_manifest.json'
_V55_CKPTS = sorted(_V55_OUT.glob('v16_slothead_f*.pt'))
if not _V55_MANIFEST.is_file():
    raise RuntimeError('TRAIN failed: v16_manifest.json was not produced')
if len(_V55_CKPTS) < 4:
    raise RuntimeError(f'TRAIN failed: only {len(_V55_CKPTS)} slot-head checkpoints were produced')
_V55_META = _v55_json.loads(_V55_MANIFEST.read_text(encoding='utf-8'))
if _V55_META.get('artifact_type') != 'rsna-knee-dinov3-slothead-v16':
    raise RuntimeError('TRAIN failed: artifact contract mismatch')
(_V55_OUT / 'V55_TRAIN_OUTPUT_READY.txt').write_text(
    'Attach this notebook Output to RSNA_Knee_DINOsaur_V4_INFER_GoldOOFSlotHead.ipynb\n',
    encoding='utf-8',
)
print('[V5.5 TRAIN] OUTPUT READY')
print('folder:', _V55_OUT)
print('manifest:', _V55_MANIFEST)
print('checkpoints:', len(_V55_CKPTS))
print('deploy weights:')
for _t, _w in _V55_META.get('deploy_weights', {}).items():
    if float(_w) > 0:
        print(f'  {_t}: {float(_w):.3f}')
